# Test based on https://github.com/NathanWycoff/prox_alasso/blob/main/python/ada_sim.py

In [1]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LassoCV
from adaptivelasso.skcompat import MAPAdaptiveLasso, MAPAdaptiveLassoCV
from adaptivelasso.utils import generate_data, report_model_results
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import PolynomialFeatures


## Compare MAPAdaptiveLasso with LassoCV

In [2]:
# Example for normal likelihood
N, P = 1000, 20
beta = np.zeros(P)
beta[:5] = 2  # 5 informative features
sigma = 1.0
lik = 'normal'

X, y = generate_data(lik, N, P, beta=beta, sigma=sigma, random_state=42)

# Split and fit as before...
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

pipe = Pipeline([
    ('scale', StandardScaler()),
    ('jax_model', MAPAdaptiveLasso(lam_prior='adaptive', lik=lik, max_iters=5000, track=False, tol=1e-4, cv=5))
])
pipe.fit(X_train, y_train)

# Use the reporting function
report_model_results(pipe.named_steps['jax_model'], X_test, y_test, lik)

# Use a pipeline for scaling and LassoCV
pipe_lassocv = Pipeline([
    ('scale', StandardScaler()),
    ('lasso', LassoCV(cv=5, random_state=0, max_iter=5000, tol=1e-4))
])
pipe_lassocv.fit(X_train, y_train)

# Use the reporting function
report_model_results(pipe_lassocv.named_steps['lasso'], X_test, y_test, lik=lik)

ES stop!
ES stop!
ES stop!
ES stop!
Estimated coefficients: [ 2.00685724  2.02559701  2.09333333  1.94567074  1.95796438  0.06564284
  0.09271615 -0.00266304  0.0441366   0.01521614  0.         -0.01378768
  0.01206377  0.06819578  0.08636152  0.05935095  0.01470796  0.01180718
 -0.04132121  0.02082752]
R²: 0.9504
RMSE: 0.9682
Estimated error variance: 0.9373983708654283
Estimated coefficients: [ 1.96264089  2.01882337  2.0179528   1.93265903  1.90844035  0.01096407
  0.03442297 -0.          0.         -0.         -0.         -0.
  0.          0.          0.0219134   0.00486638  0.          0.
 -0.          0.0357031 ]
R²: 0.9528
RMSE: 0.9444
Estimated error variance: 0.8911630396832502


{'coefs': array([ 1.96264089,  2.01882337,  2.0179528 ,  1.93265903,  1.90844035,
         0.01096407,  0.03442297, -0.        ,  0.        , -0.        ,
        -0.        , -0.        ,  0.        ,  0.        ,  0.0219134 ,
         0.00486638,  0.        ,  0.        , -0.        ,  0.0357031 ]),
 'R2': 0.9527899072830069,
 'RMSE': np.float64(0.9444386366702587),
 'MAD': None}

In [4]:


# Assume generate_data and report_model_results are already defined as above

likelihoods = ['normal', 'cauchy', 'bernoulli', 'poisson', 'zinb', 'nb']
results = []
max_iters = 3000  # Set a maximum number of iterations
tol = 1e-4  # Set a tolerance for convergence
cv = 5

for lik in likelihoods:
    print(f"Testing likelihood: {lik}")
    N, P = 1000, 20
    true_coefs = np.zeros(P)
    true_coefs[:3] = 1.0
    sigma = 1.0

    X, y = generate_data(lik, N, P, beta=true_coefs, sigma=sigma, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

    # MAPAdaptiveLasso
    pipe = Pipeline([
        ('scale', StandardScaler()),
        ('jax_model', MAPAdaptiveLasso(lam_prior='adaptive', lik=lik, max_iters=max_iters, track=True, tol=tol, cv=cv))
    ])
    pipe.fit(X_train, y_train)
    adapt_metrics = report_model_results(pipe.named_steps['jax_model'], X_test, y_test, lik)

    # LassoCV
    pipe_lasso = Pipeline([
        ('scale', StandardScaler()),
        ('lasso', LassoCV(cv=5, random_state=0, max_iter=max_iters))
    ])
    pipe_lasso.fit(X_train, y_train)
    lasso_metrics = report_model_results(pipe_lasso.named_steps['lasso'], X_test, y_test, lik)

    # Decide which is better for each metric
    if lik == 'cauchy':
        better_mad = (
            'AdaptiveLasso' if adapt_metrics['MAD'] < lasso_metrics['MAD'] else
            'LassoCV' if lasso_metrics['MAD'] < adapt_metrics['MAD'] else
            'Tie'
        )
        better_rmse = np.nan
    else:
        better_mad = np.nan
        better_rmse = (
            'AdaptiveLasso' if adapt_metrics['RMSE'] < lasso_metrics['RMSE'] else
            'LassoCV' if lasso_metrics['RMSE'] < adapt_metrics['RMSE'] else
            'Tie'
        )

    if np.isnan(lasso_metrics['R2']):
        better_r2 = 'AdaptiveLasso only'
    else:
        better_r2 = (
            'AdaptiveLasso' if adapt_metrics['R2'] > lasso_metrics['R2'] else
            'LassoCV' if lasso_metrics['R2'] > adapt_metrics['R2'] else
            'Tie'
        )

    results.append({
        'likelihood': lik,
        'R2_AdaptiveLasso': adapt_metrics['R2'],
        'RMSE_AdaptiveLasso': adapt_metrics['RMSE'],
        'MAD_AdaptiveLasso': adapt_metrics['MAD'],
        'R2_LassoCV': lasso_metrics['R2'],
        'RMSE_LassoCV': lasso_metrics['RMSE'],
        'MAD_LassoCV': lasso_metrics['MAD'],
        'Better_R2': better_r2,
        'Better_RMSE': better_rmse,
        'Better_MAD': better_mad,
    })

df_results = pd.DataFrame(results)
print(df_results)

Testing likelihood: normal
ES stop!
ES stop!
ES stop!
ES stop!
Estimated coefficients: [ 1.04177314  1.06506323  1.00960306 -0.01622986 -0.01914712  0.02094688
  0.07715846  0.00134746 -0.01181379 -0.02115135  0.03019611 -0.048533
  0.00401244  0.03012038  0.02391426  0.06165883 -0.00468517  0.01157192
 -0.0174364   0.05880315]
R²: 0.7328
RMSE: 0.9434
Estimated error variance: 0.889046835808645
Estimated coefficients: [ 0.95982998  0.99692659  1.00861647 -0.         -0.          0.
  0.02881936 -0.          0.         -0.         -0.         -0.
 -0.          0.          0.00593973  0.          0.          0.
 -0.          0.02557177]
R²: 0.7370
RMSE: 0.9360
Estimated error variance: 0.8745377131674651
Testing likelihood: cauchy
ES stop!
ES stop!
ES stop!
Estimated coefficients: [ 1.04944012  1.05288893  0.89940316  0.0857037  -0.0947982   0.050131
 -0.03410379 -0.02627072 -0.04953316 -0.06135845 -0.          0.03783665
  0.01171731 -0.08662611 -0.01989594 -0.08019521 -0.14025776 -0.04

## Test MAPAdaptiveLassoCV

In [7]:
# Example: grid search over tau0 and lr_pre for normal likelihood
N, P = 1000, 20
beta = np.zeros(P)
beta[:5] = 2
sigma = 1.0
lik = 'normal'

X, y = generate_data(lik, N, P, beta=beta, sigma=sigma, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

param_grid = {
    'tau0': [0.1, 1.0, 10.0],
    'lr_pre': [0.01, 0.1],
    'max_iters': [1000]
}

pipe = Pipeline([
    ('scale', StandardScaler()),
    ('alasso_cv', MAPAdaptiveLassoCV(param_grid=param_grid, n_splits=5, scoring='r2', random_state=42, verbose=True))
])
pipe.fit(X_train, y_train)

# Use the reporting function
metrics = report_model_results(pipe.named_steps['alasso_cv'], X_test, y_test, lik)
print("MAPAdaptiveLassoCV test metrics:", metrics)
print("Best params:", pipe.named_steps['alasso_cv'].best_params_)

Testing params: {'lr_pre': 0.01, 'max_iters': 1000, 'tau0': 0.1}
Testing params: {'lr_pre': 0.01, 'max_iters': 1000, 'tau0': 1.0}
Testing params: {'lr_pre': 0.01, 'max_iters': 1000, 'tau0': 10.0}
Testing params: {'lr_pre': 0.1, 'max_iters': 1000, 'tau0': 0.1}
ES stop!
ES stop!
ES stop!
ES stop!
Testing params: {'lr_pre': 0.1, 'max_iters': 1000, 'tau0': 1.0}
ES stop!
ES stop!
ES stop!
ES stop!
ES stop!
Testing params: {'lr_pre': 0.1, 'max_iters': 1000, 'tau0': 10.0}
ES stop!
ES stop!
ES stop!
ES stop!
Estimated coefficients: [ 2.00543513e+00  2.04933105e+00  2.04014341e+00  1.97288063e+00
  1.96716304e+00  1.88197253e-02  5.27500737e-02 -1.81425359e-02
  2.49788296e-02 -0.00000000e+00 -0.00000000e+00 -2.79504196e-02
  0.00000000e+00  0.00000000e+00  3.26812768e-02  4.16554667e-02
  4.18273680e-03  0.00000000e+00 -2.01659300e-03  6.51785338e-02]
R²: 0.9526
RMSE: 0.9460
Estimated error variance: 0.893101868581639
MAPAdaptiveLassoCV test metrics: {'coefs': array([ 2.00543513e+00,  2.049331

## Test will real data -- UCI Bike Sharing Dataset

In [2]:
df = pd.read_csv('/Users/u543860/dataset_for_adaptivelasso/bike+sharing+dataset/day.csv')
print(df.head())
df.shape

   instant      dteday  season  yr  mnth  holiday  weekday  workingday  \
0        1  2011-01-01       1   0     1        0        6           0   
1        2  2011-01-02       1   0     1        0        0           0   
2        3  2011-01-03       1   0     1        0        1           1   
3        4  2011-01-04       1   0     1        0        2           1   
4        5  2011-01-05       1   0     1        0        3           1   

   weathersit      temp     atemp       hum  windspeed  casual  registered  \
0           2  0.344167  0.363625  0.805833   0.160446     331         654   
1           2  0.363478  0.353739  0.696087   0.248539     131         670   
2           1  0.196364  0.189405  0.437273   0.248309     120        1229   
3           1  0.200000  0.212122  0.590435   0.160296     108        1454   
4           1  0.226957  0.229270  0.436957   0.186900      82        1518   

    cnt  
0   985  
1   801  
2  1349  
3  1562  
4  1600  


(731, 16)

In [4]:
X = df.drop(columns=['instant', 'dteday', 'casual', 'registered', 'cnt'])
y = df['cnt']
# One-hot Encode Categorical Variables
X = pd.get_dummies(X, columns=['season', 'mnth', 'weekday', 'weathersit'], drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

X_train = X_train.fillna(X_train.mean())
X_test = X_test.fillna(X_test.mean())
y_train = y_train.fillna(y_train.mean())
y_test = y_test.fillna(y_test.mean())

X_train = X_train.astype(float)
X_test = X_test.astype(float)

In [5]:
print(X_train.isnull().sum().sum(), X_test.isnull().sum().sum())
print(y_train.isnull().sum(), y_test.isnull().sum())

0 0
0 0


In [6]:
# Example usage in your notebook
alasso_cv = MAPAdaptiveLassoCV(
    param_grid={
        'tau0': [0.01],
        'max_iters': [1000],
        'lr_pre': [0.001],
        'mb_size': [None, 32],
        'invlam': [True, False],
        'logprox': [False],
        'lam_prior': ['adaptive'],
        'lik': ['nb', 'normal'],
        'expand': ['none', 'intr', 'quad'],  # Test all expansion options safely
        'tol': [1e-4]
    },
    n_splits=5, scoring=None,  # Auto-select scoring based on likelihood
    random_state=42, verbose=True
)

# Convert data to numpy once to avoid any DataFrame indexing issues
X_train_np = X_train.to_numpy() if hasattr(X_train, 'to_numpy') else X_train
y_train_np = y_train.to_numpy() if hasattr(y_train, 'to_numpy') else y_train

alasso_cv.fit(X_train_np, y_train_np)
best_lik = alasso_cv.best_params_['lik']
metrics = report_model_results(alasso_cv, X_test, y_test, lik=best_lik)
print("MAPAdaptiveLassoCV test metrics:", metrics)
print("Best params:", alasso_cv.best_params_)

Testing params: {'expand': 'none', 'invlam': True, 'lam_prior': 'adaptive', 'lik': 'nb', 'logprox': False, 'lr_pre': 0.001, 'max_iters': 1000, 'mb_size': None, 'tau0': 0.01, 'tol': 0.0001}
Testing params: {'expand': 'intr', 'invlam': True, 'lam_prior': 'adaptive', 'lik': 'nb', 'logprox': False, 'lr_pre': 0.001, 'max_iters': 1000, 'mb_size': None, 'tau0': 0.01, 'tol': 0.0001}
Testing params: {'expand': 'intr', 'invlam': True, 'lam_prior': 'adaptive', 'lik': 'nb', 'logprox': False, 'lr_pre': 0.001, 'max_iters': 1000, 'mb_size': 32, 'tau0': 0.01, 'tol': 0.0001}
Testing params: {'expand': 'none', 'invlam': True, 'lam_prior': 'adaptive', 'lik': 'normal', 'logprox': False, 'lr_pre': 0.001, 'max_iters': 1000, 'mb_size': 32, 'tau0': 0.01, 'tol': 0.0001}
Testing params: {'expand': 'none', 'invlam': True, 'lam_prior': 'adaptive', 'lik': 'nb', 'logprox': False, 'lr_pre': 0.001, 'max_iters': 1000, 'mb_size': 32, 'tau0': 0.01, 'tol': 0.0001}
Testing params: {'expand': 'none', 'invlam': False, 'lam_

In [7]:
# prev Best params: {'expand': 'none', 'invlam': False, 'lam_prior': 'adaptive', 'lik': 'nb', 'logprox': False, 'lr_pre': 0.001, 'max_iters': 3000, 'mb_size': 32, 'tau0': 0.1, 'tol': 0.0001}

pipe = Pipeline([
    ('alasso_cv', MAPAdaptiveLassoCV(
        param_grid={
            'tau0': [0.01],
            'max_iters': [1000],
            'lr_pre': [0.001],
            'mb_size': [None, 32],
            'invlam': [True, False],
            'logprox': [False],
            'lam_prior': ['adaptive'],
            'lik': ['nb', 'normal'],
            'expand': ['none', 'intr', 'quad'],  # Let the model handle expansion
            'tol': [1e-4]
        },
        n_splits=5, scoring='r2', random_state=42, verbose=True
    ))
])

# Fit
pipe.fit(X_train, y_train)

# Get best likelihood
best_lik = pipe.named_steps['alasso_cv'].best_params_['lik']
print("Best likelihood:", best_lik)

# Report
metrics = report_model_results(pipe.named_steps['alasso_cv'], X_test, y_test, lik=best_lik)
print("MAPAdaptiveLassoCV test metrics:", metrics)
print("Best params:", pipe.named_steps['alasso_cv'].best_params_)

Testing params: {'expand': 'none', 'invlam': True, 'lam_prior': 'adaptive', 'lik': 'nb', 'logprox': False, 'lr_pre': 0.001, 'max_iters': 1000, 'mb_size': 32, 'tau0': 0.01, 'tol': 0.0001}
Testing params: {'expand': 'intr', 'invlam': True, 'lam_prior': 'adaptive', 'lik': 'normal', 'logprox': False, 'lr_pre': 0.001, 'max_iters': 1000, 'mb_size': 32, 'tau0': 0.01, 'tol': 0.0001}
Testing params: {'expand': 'none', 'invlam': True, 'lam_prior': 'adaptive', 'lik': 'nb', 'logprox': False, 'lr_pre': 0.001, 'max_iters': 1000, 'mb_size': None, 'tau0': 0.01, 'tol': 0.0001}
Testing params: {'expand': 'none', 'invlam': False, 'lam_prior': 'adaptive', 'lik': 'normal', 'logprox': False, 'lr_pre': 0.001, 'max_iters': 1000, 'mb_size': None, 'tau0': 0.01, 'tol': 0.0001}
Testing params: {'expand': 'none', 'invlam': True, 'lam_prior': 'adaptive', 'lik': 'normal', 'logprox': False, 'lr_pre': 0.001, 'max_iters': 1000, 'mb_size': None, 'tau0': 0.01, 'tol': 0.0001}
Testing params: {'expand': 'none', 'invlam': F

TypeError: dot_general requires contracting dimensions to have the same shape, got (29,) and (435,).

In [ ]:
## Test MAPAdaptiveLasso with CV param 

In [18]:
pipe = Pipeline([
    ("scale", StandardScaler()),
    ("jax_model", MAPAdaptiveLasso(
        lam_prior="adaptive",
        lik="nb",
        max_iters=5000,
        tol=1e-4,
        track=False,
        cv=5
    ))
])

pipe.fit(X_train, y_train)
X_test_np = np.asarray(X_test).astype(np.float32)
report_model_results(pipe.named_steps["jax_model"], X_test_np, y_test, lik="nb")

ES stop!
ES stop!
ES stop!
ES stop!
Estimated coefficients: [ 0.22386099 -0.01850342  0.00940865  0.16778441  0.16663227 -0.07352877
 -0.05118984  0.06994376  0.03297109  0.12778623  0.00321629  0.0352446
  0.02521102  0.03904332  0.         -0.01473961  0.01688663  0.0585996
  0.04063061  0.04788822  0.04723228  0.          0.00104247 -0.00238854
 -0.          0.01357843  0.01239085 -0.02272922 -0.11191669]
R²: 0.2770
RMSE: 1753.5763
Mean predicted count: 5590.7184


{'coefs': array([ 0.22386099, -0.01850342,  0.00940865,  0.16778441,  0.16663227,
        -0.07352877, -0.05118984,  0.06994376,  0.03297109,  0.12778623,
         0.00321629,  0.0352446 ,  0.02521102,  0.03904332,  0.        ,
        -0.01473961,  0.01688663,  0.0585996 ,  0.04063061,  0.04788822,
         0.04723228,  0.        ,  0.00104247, -0.00238854, -0.        ,
         0.01357843,  0.01239085, -0.02272922, -0.11191669]),
 'R2': 0.276998241981244,
 'RMSE': np.float64(1753.5762714330194),
 'MAD': None}

In [22]:
# Pipeline: StandardScaler + LassoCV
lasso_pipe = Pipeline([
    ('scale', StandardScaler()),
    ('lasso', LassoCV(cv=5, random_state=42, n_jobs=-1))
])

# Fit
lasso_pipe.fit(X_train, y_train)

# result
metrics = report_model_results(lasso_pipe, X_test, y_test, lik='normal')
print("LassoCV test metrics:", metrics)

Estimated coefficients: None
R²: 0.8507
RMSE: 796.7359
Estimated error variance: 632624.4149507963
LassoCV test metrics: {'coefs': None, 'R2': 0.8507484518244175, 'RMSE': np.float64(796.735935724453), 'MAD': None}
